# Taller 02: Poda Alfa Beta 

### Grupo: Natalia Carpintero, Paula Núñez e Isabella Arrieta.

**Objetivo:** Diseñe e implemente un algoritmo de Poda Alfa Beta que se pueda aplicar de forma genérica a un árbol presentado usando listas de Python.

In [ ]:
from math import inf


def is_leaf(node):
    """Retorna True si el nodo es una hoja (valor numérico)."""
    return not isinstance(node, list)


def minimax(tree, depth=0):
    """Minimax sin poda. Devuelve el valor y la mejor secuencia."""
    if is_leaf(tree):
        return {"value": tree, "sequence": []}

    if depth % 2 == 0:  # MAX
        best_value = -inf
        best_sequence = []
        for i, child in enumerate(tree):
            result = minimax(child, depth + 1)
            candidate = result["value"]
            if candidate > best_value:
                best_value = candidate
                best_sequence = [i] + result["sequence"]
        return {"value": best_value, "sequence": best_sequence}

    # MIN
    best_value = inf
    best_sequence = []
    for i, child in enumerate(tree):
        result = minimax(child, depth + 1)
        candidate = result["value"]
        if candidate < best_value:
            best_value = candidate
            best_sequence = [i] + result["sequence"]
    return {"value": best_value, "sequence": best_sequence}


def alpha_beta(tree, depth=0, alpha=-inf, beta=inf, path=None, pruned=None):
    """Algoritmo de Poda Alfa-Beta para árboles de listas anidadas."""
    if path is None:
        path = []
    if pruned is None:
        pruned = []

    if is_leaf(tree):
        return {
            "value": tree,
            "sequence": path,
            "alpha": alpha,
            "beta": beta,
            "pruned": pruned.copy(),
            "node_type": "LEAF",
        }

    if depth % 2 == 0:  # MAX
        best_value = -inf
        best_sequence = path[:]
        for i, child in enumerate(tree):
            child_path = path + [i]
            result = alpha_beta(child, depth + 1, alpha, beta, child_path, pruned)
            candidate = result["value"]
            if candidate > best_value:
                best_value = candidate
                best_sequence = result["sequence"]

            alpha = max(alpha, best_value)
            if alpha >= beta:
                pruned.append({
                    "depth": depth,
                    "node_type": "MAX",
                    "branch_index": i,
                    "path": child_path,
                    "reason": "MAX corta por beta",
                    "alpha": alpha,
                    "beta": beta,
                })
                break

        return {
            "value": best_value,
            "sequence": best_sequence,
            "alpha": alpha,
            "beta": beta,
            "pruned": pruned.copy(),
            "node_type": "MAX",
        }

    # MIN
    best_value = inf
    best_sequence = path[:]
    for i, child in enumerate(tree):
        child_path = path + [i]
        result = alpha_beta(child, depth + 1, alpha, beta, child_path, pruned)
        candidate = result["value"]
        if candidate < best_value:
            best_value = candidate
            best_sequence = result["sequence"]

        beta = min(beta, best_value)
        if alpha >= beta:
            pruned.append({
                "depth": depth,
                "node_type": "MIN",
                "branch_index": i,
                "path": child_path,
                "reason": "MIN corta por alpha",
                "alpha": alpha,
                "beta": beta,
            })
            break

    return {
        "value": best_value,
        "sequence": best_sequence,
        "alpha": alpha,
        "beta": beta,
        "pruned": pruned.copy(),
        "node_type": "MIN",
    }


def alpha_beta_pruning(tree):
    """Función principal para aplicar poda alfa-beta a cualquier árbol anidado."""
    result = alpha_beta(tree, depth=0, alpha=-inf, beta=inf, path=[], pruned=[])
    return {
        "valor": result["value"],
        "secuencia_optima": result["sequence"],
        "alfa_final": result["alpha"],
        "beta_final": result["beta"],
        "ramas_podadas": result["pruned"],
    }


# ----------- Ejemplo de prueba -----------

tree = [[3, -5, 2], [-5, -7, 4], [-9, 6, -8]]

minimax_result = minimax(tree)
alpha_beta_result = alpha_beta_pruning(tree)

print("Minimax:", minimax_result)
print("Alpha-Beta:", alpha_beta_result)

# Validación rápida
assert minimax_result["value"] == -5
assert alpha_beta_result["valor"] == -5
assert alpha_beta_result["secuencia_optima"] == [0, 1]
